# 02 Silver: clean and standardise

**Goal:** one trusted row per month and pollutant, with proper types.

Business rules applied here:
1. Parse `reading_date` to a real `DATE`.
2. Standardise pollutant codes (`PM 10` becomes `PM10`, `PM 2.5` becomes `PM2.5`).
3. Turn empty concentration strings into `NULL` and flag them with `is_missing` (we never invent values).
4. Keep only one row per (month, pollutant): the most recently ingested one.

In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA air_quality")

DataFrame[]

In [0]:
%sql
CREATE OR REPLACE TABLE silver_air_pollution AS
SELECT
  TO_DATE(TRIM(reading_date))                                   AS reading_month,
  REPLACE(UPPER(TRIM(pollutant)), ' ', '')                      AS pollutant,
  CAST(NULLIF(TRIM(concentration), '') AS DOUBLE)               AS concentration,
  NULLIF(TRIM(concentration), '') IS NULL                       AS is_missing,
  _ingested_at,
  _source_file
FROM bronze_air_pollution
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY TO_DATE(TRIM(reading_date)), REPLACE(UPPER(TRIM(pollutant)), ' ', '')
  ORDER BY _ingested_at DESC
) = 1

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT pollutant,
       COUNT(*)                                   AS months,
       SUM(CASE WHEN is_missing THEN 1 ELSE 0 END) AS missing_months,
       MIN(reading_month)                         AS first_month,
       MAX(reading_month)                         AS last_month
FROM silver_air_pollution
GROUP BY pollutant
ORDER BY pollutant

pollutant,months,missing_months,first_month,last_month
CO,96,14,2017-01-01,2024-12-01
NO2,96,2,2017-01-01,2024-12-01
O3,96,2,2017-01-01,2024-12-01
PM10,96,2,2017-01-01,2024-12-01
PM2.5,96,12,2017-01-01,2024-12-01
SO2,96,2,2017-01-01,2024-12-01
